In [17]:
import pandas as pd

## Refactored/Modular Crosswalk (WIP) Assisted by Claude

In [18]:
# %% Structural constants (crosswalk column names, not construct-specific) --
 
CODE_COLS = {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title', 'OEWS Code',
             'OEWS Title', 'CPS Code', 'CPS Title', 'manual_soc_to_cps'}

In [19]:
# %% Reference data (crosswalk + BLS weights + BLS employment) -------------
 
def load_reference_data(crosswalk_path='../data/crosswalks/final_crosswalk.csv',
                         oews_weights_html_path="../data/crosswalks/Mapping Employment Projections and O_NET data_ a methodological overview _ Monthly Labor Review _ U.S. Bureau of Labor Statistics.html",
                         oews_path='../data/national_M2024_dl.xlsx'):
    """Load the crosswalk, OEWS aggregate weights, and OEWS employment totals.
    These are structural (same for every construct)."""
    crosswalk = pd.read_csv(crosswalk_path)
    oews_weights = pd.read_html(oews_weights_html_path)[2]
    oews = pd.read_excel(oews_path, sheet_name='national_M2024_dl', dtype=str)
    return crosswalk, oews_weights, oews
 
 
def load_sample(occ_level_path='../data/occ_level.csv'):
    """Load the file that defines the 'normal sample' of O*NET-SOC codes
    (currently the Eloundou score file). Any construct gets restricted to
    the codes present here."""
    df_raw = pd.read_csv(occ_level_path)
    df_raw = df_raw.drop(columns=['Title'])
    return df_raw

In [20]:
# %% Plug in any O*NET-level construct, restricted to the sample -----------
 
# Download once and point to it locally:
# O*NET-SOC 2010 -> O*NET-SOC 2019: https://www.onetcenter.org/taxonomy/2019/walk/2010_to_2019_Crosswalk.csv?fmt=csv
def load_2010_to_2019_crosswalk(path='../data/crosswalks/2010_to_2019_Crosswalk.csv'):
    xwalk = pd.read_csv(path)
    xwalk = xwalk.rename(columns={
        'O*NET-SOC 2010 Code': 'onet_2010',
        'O*NET-SOC 2019 Code': 'onet_2019',
    })
    return xwalk[['onet_2010', 'onet_2019']]
 
 
def harmonize_2010_onet(construct, onet_col, index_cols, xwalk_2010_to_2019):
    """Walk a 2010-vintage O*NET construct forward to O*NET-SOC 2019 codes.
    Simple mean / simple replace: groupby + mean on the 2019 code handles
    both splits (one row per group, value passes through unchanged) and
    merges (multiple rows per group, averaged) in one step."""
    merged = construct.merge(xwalk_2010_to_2019, left_on=onet_col, right_on='onet_2010', how='left')
    harmonized = merged.groupby('onet_2019', as_index=False)[index_cols].mean()
    harmonized = harmonized.rename(columns={'onet_2019': 'O*NET-SOC Code'})
    return harmonized
 
 
def load_onet_construct(path, onet_col_in_file, sample_codes, index_cols=None,
                         rename_to='O*NET-SOC Code', read_kwargs=None,
                         vintage='2019', xwalk_2010_to_2019=None):
    """Load any O*NET-level dataset and restrict it to the codes in sample_codes.
 
    e.g. for telework (2010 O*NET vintage, needs harmonizing first):
        xwalk = load_2010_to_2019_crosswalk()
        tele = load_onet_construct('../data/occupations_workathome.csv',
                                    onet_col_in_file='onetsoccode',
                                    sample_codes=df_raw['O*NET-SOC Code'],
                                    index_cols=['teleworkable'],
                                    vintage='2010',
                                    xwalk_2010_to_2019=xwalk)
        INDEX_COLS = ['teleworkable']
 
    vintage='2019' (default) skips harmonization entirely - use this when the
    construct is already on the current O*NET-SOC taxonomy.
    """
    read_kwargs = read_kwargs or {}
    construct = pd.read_csv(path, **read_kwargs)
 
    if vintage == '2010':
        construct = harmonize_2010_onet(construct, onet_col_in_file, index_cols, xwalk_2010_to_2019)
        onet_col_in_file = 'O*NET-SOC Code'  # already renamed inside harmonize_2010_onet
 
    construct = construct[construct[onet_col_in_file].isin(sample_codes)]
    construct = construct.rename(columns={onet_col_in_file: rename_to})
    return construct
 
 

In [21]:
# %% ONET -> SOC --------------------------------------------------------
 
def merge_to_crosswalk(construct_df, crosswalk, onet_col='O*NET-SOC Code'):
    """Attach SOC/OEWS/CPS codes from the finalized crosswalk to a construct."""
    df = construct_df.merge(crosswalk,
                             left_on=onet_col,
                             right_on='ONET Code',
                             how='left',
                             indicator=True)
 
    assert (df["_merge"] == "left_only").sum() == 0, "Some O*NET codes failed to merge"
 
    df = df.drop(columns=[onet_col, '_merge'], errors='ignore')
    if onet_col != 'O*NET-SOC Code' and 'O*NET-SOC Code' in df.columns:
        df = df.drop(columns=['O*NET-SOC Code'])
    return df

def collapse_to_soc(df, index_cols, temp_dir='../temp', name='construct'):
    """Collapse a construct down to the 6-digit SOC level."""
    df = df.copy()
    df['agg_level_ONET'] = df.duplicated(subset=['SOC Code'], keep=False)
 
    std_df_onet = df.groupby('SOC Code')[index_cols].std().reset_index()
    std_df_onet = std_df_onet.rename(columns={col: f'{col}_std_ONET' for col in index_cols})
 
    df_onet_with_stats = df.merge(std_df_onet, on='SOC Code', how='left')
 
    collapsed_rows_onet = df_onet_with_stats[df_onet_with_stats['agg_level_ONET'] == True]
    collapsed_rows_onet.to_csv(f'{temp_dir}/{name}_collapsed_at_ONET.csv', index=False)
 
    code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title'}
 
    agg_dict = {col: 'mean' for col in index_cols}
    agg_dict.update({col: 'first' for col in code_cols_for_agg if col in df_onet_with_stats.columns})
    agg_dict.update({f'{col}_std_ONET': 'first' for col in index_cols})
    agg_dict.update({'agg_level_ONET': 'first'})
 
    df_soc = df_onet_with_stats.groupby('SOC Code', as_index=False).agg(agg_dict)
 
    assert len(df_soc) <= df['SOC Code'].nunique()
    assert df_soc['SOC Code'].is_unique, "duplicate SOC Codes survived the groupby"
 
    return df_soc
 

In [22]:
# %% Diagnostics ---------------------------------------------------------
 
def diagnostics_mean_shift(df_before, df_after, index_cols, drop_codes=None):
    """Compare means of index_cols before/after an aggregation step.
    Pass drop_codes (list of SOC codes) to also show the shift excluding
    a few high-variance codes, same as the original notebook check."""
    rows = []
    for col in index_cols:
        rows.append({
            'column': col,
            'mean_before': df_before[col].mean(),
            'mean_after': df_after[col].mean(),
            'diff': df_after[col].mean() - df_before[col].mean(),
        })
    diag = pd.DataFrame(rows)
 
    if drop_codes:
        code_col = 'SOC Code' if 'SOC Code' in df_before.columns else None
        if code_col:
            temp_before = df_before[~df_before[code_col].isin(drop_codes)]
            temp_after = df_after[~df_after[code_col].isin(drop_codes)]
            for i, col in enumerate(index_cols):
                diag.loc[i, 'mean_before_excl'] = temp_before[col].mean()
                diag.loc[i, 'mean_after_excl'] = temp_after[col].mean()
                diag.loc[i, 'diff_excl'] = temp_after[col].mean() - temp_before[col].mean()
 
    return diag

In [23]:
# %% Split codes that map 1:1 to CPS vs. those that need collapsing --------
 
def split_mapped_unmapped(df_soc):
    df_soc = df_soc.copy()
    df_soc["SOC_per_CPS"] = df_soc.groupby("CPS Code")["CPS Code"].transform("size")
 
    is_mapped = df_soc["SOC_per_CPS"] <= 1
 
    mapped = df_soc[is_mapped]
 
    cols_to_drop = CODE_COLS - {'CPS Code', 'CPS Title'}
    mapped = mapped.drop(columns=cols_to_drop, errors="ignore")
    mapped['agg_level_SOC'] = False
    mapped['agg_level_OEWS'] = False
 
    to_map = df_soc[~is_mapped]
 
    return mapped, to_map

In [24]:
def compute_oews_weights(to_map, oews_weights, expected_matches=None):
    """Attach BLS 2018 SOC weights to the SOC codes that still need collapsing.
 
    39-1013 had to be merged to one SOC code from 2 in the 2010->2018 shift.
    The merged cell appears as a duplicate (in the BLS table); it maps 1:1 to OEWS so dropping it.
    """
    oews_weights_clean = oews_weights.drop_duplicates(subset='2018 SOC code', keep='first')
 
    to_map_with_weights = to_map.merge(oews_weights_clean[['2018 SOC code', '2018 weight']],
                                        left_on="SOC Code",
                                        right_on="2018 SOC code",
                                        how="left",
                                        indicator=True)
 
    actual_matches = (to_map_with_weights['_merge'] == 'both').sum()
    if expected_matches is None:
        expected_matches = to_map['SOC Code'].isin(oews_weights_clean['2018 SOC code']).sum()
    assert actual_matches == expected_matches, (
        f"Incorrect number merged at this level: expected {expected_matches}, got {actual_matches}"
    )
 
    to_map_with_weights = to_map_with_weights.drop(columns=["2018 SOC code", "_merge"])
 
    to_map_with_weights['2018 weight'] = pd.to_numeric(
        to_map_with_weights['2018 weight'],
        errors='coerce'
    )
 
    group_size = to_map_with_weights.groupby('OEWS Code')['SOC Code'].transform('size')
    group_weight_sum = to_map_with_weights.groupby('OEWS Code')['2018 weight'].transform('sum')
 
    # If the group's weights don't cover at least 98%, treat the whole group as
    # unreliable and fall back to a simple average for every member
    low_coverage = group_weight_sum < 0.98
    to_map_with_weights.loc[low_coverage, '2018 weight'] = 1 / group_size[low_coverage]
 
    to_map_with_weights['2018 weight'] = to_map_with_weights['2018 weight'].fillna(1 / group_size)
 
    return to_map_with_weights
 
 
def collapse_to_oews(to_map_with_weights, index_cols, temp_dir='../temp', name='construct'):
    """Collapse a construct down to the OEWS level using the BLS weights."""
    to_map_with_weights = to_map_with_weights.copy()
    to_map_with_weights['agg_level_SOC'] = to_map_with_weights.duplicated(subset=['OEWS Code'], keep=False)
 
    std_df_soc = to_map_with_weights.groupby('OEWS Code')[index_cols].std().reset_index()
    std_df_soc = std_df_soc.rename(columns={col: f'{col}_std_SOC' for col in index_cols})
 
    df_soc_with_stats = to_map_with_weights.merge(std_df_soc, on='OEWS Code', how='left')
 
    collapsed_rows_soc = df_soc_with_stats[df_soc_with_stats['agg_level_SOC'] == True]
    collapsed_rows_soc.to_csv(f'{temp_dir}/{name}_collapsed_at_SOC.csv', index=False)
 
    code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title'}
 
    for col in index_cols:
        df_soc_with_stats[col] = pd.to_numeric(df_soc_with_stats[col], errors='coerce')
        df_soc_with_stats[f'{col}_weighted'] = df_soc_with_stats[col] * df_soc_with_stats['2018 weight']
 
    agg_dict = {f'{col}_weighted': 'sum' for col in index_cols}
    agg_dict.update({col: 'first' for col in code_cols_for_agg if col in df_soc_with_stats.columns})
    agg_dict.update({f'{col}_std_ONET': 'first' for col in index_cols if f'{col}_std_ONET' in df_soc_with_stats.columns})
    agg_dict.update({f'{col}_std_SOC': 'first' for col in index_cols})
    agg_dict.update({'agg_level_ONET': 'first'})
    agg_dict.update({'agg_level_SOC': 'first'})
    agg_dict.update({'SOC_per_CPS': 'first'})
 
    df_oews = df_soc_with_stats.groupby('OEWS Code', as_index=False).agg(agg_dict)
 
    assert df_oews['OEWS Code'].is_unique, "duplicate OEWS Codes survived the groupby"
 
    df_oews = df_oews.rename(columns={f'{col}_weighted': f'{col}' for col in index_cols})
    return df_oews
 
 

In [25]:
# %% OEWS -> CPS -------------------------------------------------------------
 
def merge_oews_employment(df_oews, oews):
    """Bring in OEWS total employment before the final weighted collapse to CPS."""
    df_oews_with_emp = df_oews.merge(oews[['OCC_CODE', 'TOT_EMP']],
                                      left_on='OEWS Code',
                                      right_on='OCC_CODE',
                                      how='left',
                                      indicator=True)
 
    assert (df_oews_with_emp['_merge'] == 'left_only').sum() == 0, \
        "Some OEWS codes did not get their employment merged"
 
    df_oews_with_emp = df_oews_with_emp.drop(columns=['OCC_CODE', '_merge'])
 
    df_oews_with_emp['TOT_EMP'] = pd.to_numeric(df_oews_with_emp['TOT_EMP'])
    df_oews_with_emp['emp_share'] = df_oews_with_emp.groupby('CPS Code')['TOT_EMP'].transform(lambda x: x / x.sum())
 
    return df_oews_with_emp
 
 
def collapse_to_cps(df_oews_with_emp, index_cols, temp_dir='../temp', name='construct'):
    """Collapse a construct down to the CPS level using employment shares."""
    df_oews_with_emp = df_oews_with_emp.copy()
    df_oews_with_emp['agg_level_OEWS'] = df_oews_with_emp.duplicated(subset=['CPS Code'], keep=False)
 
    std_df_oews = df_oews_with_emp.groupby('CPS Code')[index_cols].std().reset_index()
    std_df_oews = std_df_oews.rename(columns={col: f'{col}_std_CPS' for col in index_cols})
 
    df_oews_with_stats = df_oews_with_emp.merge(std_df_oews, on='CPS Code', how='left')
 
    collapsed_rows_cps = df_oews_with_stats[df_oews_with_stats['agg_level_OEWS'] == True]
    collapsed_rows_cps.to_csv(f'{temp_dir}/{name}_collapsed_at_CPS.csv', index=False)
 
    for col in index_cols:
        df_oews_with_stats[f'{col}_std_CPS'] = df_oews_with_stats[f'{col}_std_CPS'].fillna(0)
 
    code_cols_for_agg = CODE_COLS - {'O*NET Code', 'O*NET Title', 'SOC Code', 'SOC Title',
                                      'OEWS Code', 'OEWS Title'}
 
    for col in index_cols:
        df_oews_with_stats[col] = pd.to_numeric(df_oews_with_stats[col], errors='coerce')
        df_oews_with_stats[f'{col}_weighted'] = df_oews_with_stats[col] * df_oews_with_stats['emp_share']
 
    agg_dict = {f'{col}_weighted': 'sum' for col in index_cols}
    agg_dict.update({col: 'first' for col in code_cols_for_agg if col in df_oews_with_stats.columns})
    agg_dict.update({f'{col}_std_ONET': 'first' for col in index_cols if f'{col}_std_ONET' in df_oews_with_stats.columns})
    agg_dict.update({f'{col}_std_SOC': 'first' for col in index_cols})
    agg_dict.update({f'{col}_std_CPS': 'first' for col in index_cols})
    agg_dict.update({'agg_level_ONET': 'first'})
    agg_dict.update({'agg_level_SOC': 'first'})
    agg_dict.update({'agg_level_OEWS': 'first'})
    agg_dict.update({'SOC_per_CPS': 'first'})
 
    df_cps = df_oews_with_stats.groupby('CPS Code', as_index=False).agg(agg_dict)
 
    assert df_cps['CPS Code'].is_unique, "duplicate CPS Codes survived the groupby"
 
    df_cps = df_cps.rename(columns={f'{col}_weighted': f'{col}' for col in index_cols})
    return df_cps
 
 
def finalize_cps(df_cps, mapped, df_full, index_cols):
    """Stack mapped (1:1) and collapsed CPS rows, then compute the pooled std."""
    df_cps = pd.concat([df_cps, mapped])
 
    std_df_cps = df_full.groupby('CPS Code')[index_cols].std().reset_index()
    std_df_cps = std_df_cps.rename(columns={col: f'{col}_std_pooled' for col in index_cols})
 
    df_cps_with_std = df_cps.merge(std_df_cps, on='CPS Code', how='left')
 
    for col in index_cols:
        df_cps_with_std[f'{col}_std_pooled'] = df_cps_with_std[f'{col}_std_pooled'].fillna(0)
 
    df_cps_with_std = df_cps_with_std.sort_values(by='CPS Code').reset_index(drop=True)
    return df_cps_with_std

In [26]:
# %% Top-level pipeline: plug in any construct here --------------------
 
def run_construct_pipeline(construct_df, index_cols, crosswalk, oews_weights, oews,
                            name, onet_col='O*NET-SOC Code',
                            out_dir='../data', temp_dir='../temp',
                            drop_codes_for_diagnostics=None):
    """Run a full O*NET -> SOC -> OEWS -> CPS collapse for any construct.
 
    construct_df : DataFrame with `onet_col` plus one or more columns listed
                    in index_cols (already restricted to the sample, e.g. via
                    load_onet_construct()).
    index_cols    : list of columns to aggregate, e.g. ['teleworkable'] or the
                    original ['dv_rating_alpha', 'dv_rating_beta', ...].
    name          : short label used for output filenames, e.g. 'telework'.
 
    Returns a dict with diagnostics, df_soc, and df_cps.
    """
    df = merge_to_crosswalk(construct_df, crosswalk, onet_col=onet_col)
 
    df_soc_raw = collapse_to_soc(df, index_cols, temp_dir=temp_dir, name=name)
    diag_onet_to_soc = diagnostics_mean_shift(df, df_soc_raw, index_cols,
                                               drop_codes=drop_codes_for_diagnostics)
 
    mapped, to_map = split_mapped_unmapped(df_soc_raw)
 
    to_map_with_weights = compute_oews_weights(to_map, oews_weights)
    df_oews = collapse_to_oews(to_map_with_weights, index_cols, temp_dir=temp_dir, name=name)
 
    df_oews_with_emp = merge_oews_employment(df_oews, oews)
    df_cps_collapsed = collapse_to_cps(df_oews_with_emp, index_cols, temp_dir=temp_dir, name=name)
 
    df_cps_with_std = finalize_cps(df_cps_collapsed, mapped, df, index_cols)
 
    diag_oews_to_cps = diagnostics_mean_shift(df_oews_with_emp, df_cps_collapsed, index_cols)
 
    # SOC-level export
    soc_path = f'{out_dir}/{name}_soc_level.csv'
    df_soc_raw.to_csv(soc_path, index=False)
 
    # CPS-level export
    cps_path = f'{out_dir}/{name}_cps_level.csv'
    df_cps_with_std.to_csv(cps_path, index=False)
 
    diagnostics = {
        'onet_to_soc': diag_onet_to_soc,
        'oews_to_cps': diag_oews_to_cps,
        'n_soc_codes': df_soc_raw['SOC Code'].nunique(),
        'n_cps_codes': df_cps_with_std['CPS Code'].nunique(),
        'n_collapsed_at_onet': int(df_soc_raw['agg_level_ONET'].sum()),
    }
 
    return {'diagnostics': diagnostics, 'df_soc': df_soc_raw, 'df_cps': df_cps_with_std,
            'soc_path': soc_path, 'cps_path': cps_path}


In [27]:
# %% Run --------------------------------------------------------------------
 
df_raw = load_sample()
crosswalk, oews_weights, oews = load_reference_data()
sample_codes = df_raw['O*NET-SOC Code']

In [33]:
# --- Example 1: original Eloundou exposure scores ---------------------
eloundou_index_cols = ['dv_rating_alpha', 'dv_rating_beta', 'dv_rating_gamma',
                        'human_rating_alpha', 'human_rating_beta', 'human_rating_gamma']
eloundou_construct = df_raw.rename(columns={'O*NET-SOC Code': 'O*NET-SOC Code'})

result_eloundou = run_construct_pipeline(
    eloundou_construct, eloundou_index_cols, crosswalk, oews_weights, oews,
    name='eloundou', drop_codes_for_diagnostics=['15-1211', '15-1299']
)
print(result_eloundou['diagnostics'])

{'onet_to_soc':                column  mean_before  mean_after      diff  mean_before_excl  \
0     dv_rating_alpha     0.141813    0.137903 -0.003910          0.138175   
1      dv_rating_beta     0.344732    0.325708 -0.019024          0.340275   
2     dv_rating_gamma     0.547652    0.513513 -0.034139          0.542375   
3  human_rating_alpha     0.144119    0.135505 -0.008614          0.142592   
4   human_rating_beta     0.302671    0.285667 -0.017004          0.299217   
5  human_rating_gamma     0.461223    0.435829 -0.025393          0.455841   

   mean_after_excl  diff_excl  
0         0.137282  -0.000894  
1         0.324826  -0.015449  
2         0.512371  -0.030004  
3         0.135194  -0.007398  
4         0.284939  -0.014277  
5         0.434685  -0.021157  , 'oews_to_cps':                column  mean_before  mean_after      diff
0     dv_rating_alpha     0.120068    0.123709  0.003641
1      dv_rating_beta     0.302951    0.298885 -0.004067
2     dv_rating_gamma     

In [35]:
xwalk_2010_to_2019 = load_2010_to_2019_crosswalk('../data/crosswalks/2010_to_2019_Crosswalk.csv')

tele = load_onet_construct('../data/occupations_workathome.csv',
                            onet_col_in_file='onetsoccode',
                            sample_codes=sample_codes,
                            index_cols=['teleworkable'],
                            vintage='2010',
                            xwalk_2010_to_2019=xwalk_2010_to_2019)

result_telework = run_construct_pipeline(
    tele, ['teleworkable'], crosswalk, oews_weights, oews, name='telework'
)